In [ ]:
# =================================
# 設定
# =================================
IS_ONLINE_JUDGE = False

LIMIT_QUERY_CNT = None

DEBUG = True
DEBUG_QUERY = None
DEBUG_L = None

MAX_TIME = 1.25
FILE_NUM = 100

# =================================
# 初期化
# =================================
import time

GLOBAL_START_TIME = time.perf_counter()

import random

random.seed(0)

if IS_ONLINE_JUDGE:
    DEBUG = False

# =================================
# Import
# =================================
import math
import random

# =================================
# 共通
# =================================
if True:
    from importlib import reload

    import common

    reload(common)


from common import (
    Env,
    EnvOffline,
    UnionFind,
    calc_centroid,
    calc_dist,
    calc_rectangle_area,
    construct_dag_and_rdag_from_edges,
    construct_dag_from_edges,
    construct_dist_matrix,
    construct_dist_matrix_rectangle,
    construct_graph,
    construct_sorted_edges,
    cut_graph,
    exponential_schedule,
    kruskals_algorithm,
    linear_schedule,
    my_deepcopy,
    prim,
    prim_k,
    prim_vs,
    priority_topological_sort,
    scc,
    scc_construct,
    sort_pair,
)


# =================================
# 汎用（main.py専用）
# =================================
def debug_print(*args, **kwargs):
    if IS_ONLINE_JUDGE:
        return
    print(*args, **kwargs)


def print_elapsed_time():
    debug_print(f"elapsed time: {(time.perf_counter() - GLOBAL_START_TIME) * 1000:.2f}msecs")

## クエリによる順位更新

- 制約リスト <= []
- Q 回繰り返す

  - ランダムな点で MST を得る
  - MST の 1 辺を選ぶ
    - 辺をカットする
    - 2 グループを得る
    - グループ間の全ての辺を貼る（ - 元の辺）
    - 元の辺 < 抽出した辺の制約をリストに追加する

- トポロジカルソートしながら元の順位を保持して辺をソートする
- error を計算してみる


In [408]:
def calc_edge_list(env, points):
    edge_dists = []
    edge_id = 0
    for i in range(env.N):
        for j in range(i + 1, env.N):
            dist = calc_dist(points[i], points[j])
            edge_dists.append((dist, edge_id))
            edge_id += 1
    return sorted(edge_dists, key=lambda x: x[0])


def rank_sqrt_error(rank1, rank2):
    N = len(rank1)
    rank2_num_to_rank = {}
    for i, r2 in enumerate(rank2):
        rank2_num_to_rank[r2] = i

    cost = 0
    for i, v in enumerate(rank1):
        cost += abs(i - rank2_num_to_rank[v]) ** 2

    return cost / N


def rank_error(rank1, rank2):
    N = len(rank1)
    rank2_num_to_rank = {}
    for i, r2 in enumerate(rank2):
        rank2_num_to_rank[r2] = i

    cost = 0
    for i, v in enumerate(rank1):
        cost += abs(i - rank2_num_to_rank[v])

    return cost / N

In [409]:
file_num = 0
input_file_path = f"../in/{file_num:04d}.txt"
output_file_path = f"../out/{file_num:04d}.txt"
env = EnvOffline(input_file_path, output_file_path)

edgeid_to_poinid = {}
pointid_to_edgeid = {}
edge_id = 0
for i in range(env.N):
    for j in range(i + 1, env.N):
        edgeid_to_poinid[edge_id] = (i, j)
        pointid_to_edgeid[(i, j)] = edge_id
        edge_id += 1

correct_edge_dists = calc_edge_list(env, env.coordinates)
estimate_edge_dists = calc_edge_list(env, env.cneter_points)

correct_ranks = [v[1] for v in correct_edge_dists]
estimate_ranks = [v[1] for v in estimate_edge_dists]

In [410]:
import random

constrains = set()
try:
    for _ in range(400):
        point_ids = list(range(env.N))
        random_vs = random.sample(point_ids, k=15)
        edges = env.query(random_vs)

        graph = construct_graph_from_edges(edges, random_vs)

        for e in edges:
            e1_id = pointid_to_edgeid[e]
            group_vs1, group_vs2 = cut_graph(graph, e)
            for v1 in group_vs1:
                for v2 in group_vs2:
                    sv1, sv2 = sort_pair((v1, v2))
                    if e == (sv1, sv2):
                        continue
                    e2_id = pointid_to_edgeid[(sv1, sv2)]
                    long_c1 = env.coordinates[sv1]
                    long_c2 = env.coordinates[sv2]
                    short_c1 = env.coordinates[e[0]]
                    short_c2 = env.coordinates[e[1]]
                    dist1 = calc_dist(short_c1, short_c2)
                    dist2 = calc_dist(long_c1, long_c2)

                    if dist1 >= dist2:
                        print((dist1, dist2), e, (sv1, sv2), e1_id, e2_id)
                    assert dist1 < dist2

                    if (e2_id, e1_id) not in constrains:
                        constrains.add((e1_id, e2_id))
except Exception as _:
    print("NOOOOOO")

(2385, 2385) (145, 786) (408, 786) 106055 243341
NOOOOOO


In [ ]:
def sorted_dag(
    vs: list[int],  # 頂点のリスト
    graph: dict[int, list[int]],  # 隣接リスト
    graph_r: dict[int, list[int]],  # 逆グラフ
    initial_ranking: list[int],  # 初期のランク
):
    """閉路を含むDAGを初期のランクに従ってソートする"""
    label_num, group = scc(graph, graph_r, vs)
    ssc_graph, belong_lists = scc_construct(graph, label_num, group, vs)

    num_to_ranking = {i: j for j, i in enumerate(initial_ranking)}
    construct_initial_ranking = {}
    for i, belong_list in enumerate(belong_lists):
        ssc_ranks = [num_to_ranking[v] for v in belong_list]
        n = len(ssc_ranks)
        mid_rank = sorted(ssc_ranks)[n // 2]
        construct_initial_ranking[i] = mid_rank

    result_rank_construct = priority_topological_sort(ssc_graph, construct_initial_ranking)
    result_rank = []
    for i in result_rank_construct:
        now_belong = belong_lists[i]
        result_rank.extend(now_belong)

    return result_rank

In [411]:
# import matplotlib.pyplot as plt
# import numpy as np

# tmp_vs = set()
# for e in edges:
#     a, b = e
#     tmp_vs.add(a)
#     tmp_vs.add(b)
# tmp_vs = list(tmp_vs)
# np.array(env.coordinates)[tmp_vs]

# plt.figure(figsize=(15, 15))
# edges_arr = np.array(edges)
# for e in edges_arr:
#     a, b = e
#     plt.plot(
#         [env.coordinates[a][0], env.coordinates[b][0]], [env.coordinates[a][1], env.coordinates[b][1]], color="blue"
#     )
# for v in tmp_vs:
#     plt.scatter(env.coordinates[v][0], env.coordinates[v][1], color="red")
#     plt.annotate(str(v), (env.coordinates[v][0], env.coordinates[v][1]), fontsize=15, color="red")
# plt.show()

In [412]:
# import networkx as nx

# nx_graph = nx.DiGraph(graph)
# # グラフの描画
# pos = nx.spring_layout(nx_graph)
# nx.draw(nx_graph, pos, with_labels=True, node_color="lightblue", node_size=300, font_size=10)

In [ ]:
# target_edge_ids = set()
# for e in constrains:
#     e1_id, e2_id = e
#     target_edge_ids.add(e1_id)
#     target_edge_ids.add(e2_id)

EDGE_NUM = len(estimate_edge_dists)
edge_dag = construct_dag_from_edges(constrains, list(range(EDGE_NUM)))

In [414]:
result_ranks = priority_topological_sort(edge_dag, {v: k for k, v in estimate_edge_dists})

ValueError: Graph is not a DAG or has cycles

In [ ]:
for correct, estimate, result in zip(correct_ranks, estimate_ranks, result_ranks):
    print(f"{correct} {estimate} {result}")

204318 172207 172207
253786 89397 89397
64213 88522 88522
302366 187643 187643
171090 77601 77601
305319 154676 154676
167946 171630 171630
18434 119874 119874
105677 72561 72561
64402 294084 294084
76839 289521 289521
236616 106300 106300
206038 300486 300486
7483 53801 53801
225464 297113 297113
20586 200170 200170
107812 221326 221326
125094 178037 178037
186559 18650 18650
271368 205276 205276
231719 313793 313793
277519 275141 275141
9936 168229 168229
154676 63909 63909
277 148965 148965
283472 172209 172209
156810 223022 223022
205186 256731 256731
48630 301541 301541
172340 129004 129004
261748 243918 243918
48199 212589 212589
78664 79055 79055
254401 302366 302366
72918 290921 290921
128577 91532 91532
193344 45572 45572
257784 118897 118897
259334 287913 287913
243846 246519 246519
246576 141930 141930
281060 58857 58857
123212 78664 78664
196036 134718 134718
171278 302767 302767
269504 3818 3818
290665 182258 182258
307835 186559 186559
116168 222201 222201
246379 300966 3

In [ ]:
print("四角形中心", rank_error(correct_ranks, estimate_ranks))
print("制約", rank_error(correct_ranks, result_ranks))

四角形中心 8605.000425531915
制約 8605.000425531915
